In [ ]:
# ============================================
# SMALL LLM + UNCERTAINTY ANALYSIS PROJECT
# Task: Sentiment Analysis
# Model: DistilBERT
# Dataset: IMDB
# Uncertainty:
#   1. Confidence Score
#   2. Entropy
#   3. Multi-run Stability
#   4. Hidden State Extraction
# ============================================


# ============================================
# STEP 1: INSTALL LIBRARIES
# ============================================

!pip install transformers datasets torch -q


# ============================================
# STEP 2: IMPORT LIBRARIES
# ============================================

import torch
import numpy as np
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import torch.nn.functional as F


# ============================================
# STEP 3: CHECK GPU
# ============================================

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


# ============================================
# STEP 4: LOAD MODEL
# ============================================

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

print("\nModel Loaded Successfully")


# ============================================
# STEP 5: LOAD DATASET
# ============================================

dataset = load_dataset("imdb")

print(dataset)

# Take a few samples
samples = dataset["test"][:5]

texts = samples["text"]
labels = samples["label"]


# ============================================
# STEP 6: DEFINE ENTROPY FUNCTION
# ============================================

def calculate_entropy(probs):

    probs = probs + 1e-12

    entropy = -torch.sum(probs * torch.log(probs))

    return entropy.item()


# ============================================
# STEP 7: PREDICTION FUNCTION
# ============================================

def predict_with_uncertainty(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs, output_hidden_states=True)

    logits = outputs.logits

    probs = F.softmax(logits, dim=-1)

    confidence, prediction = torch.max(probs, dim=-1)

    entropy = calculate_entropy(probs[0])

    hidden_states = outputs.hidden_states

    # Last layer embedding
    last_hidden = hidden_states[-1]

    # Embedding variance
    embedding_variance = torch.var(last_hidden).item()

    return {
        "prediction": prediction.item(),
        "confidence": confidence.item(),
        "entropy": entropy,
        "embedding_variance": embedding_variance
    }


# ============================================
# STEP 8: LABEL MAPPING
# ============================================

label_map = {
    0: "NEGATIVE",
    1: "POSITIVE"
}


# ============================================
# STEP 9: RUN EXPERIMENT
# ============================================

for i, text in enumerate(texts):

    print("\n" + "="*60)

    print(f"SAMPLE {i+1}")

    print("="*60)

    short_text = text[:300]

    print("\nTEXT:\n")
    print(short_text)

    result = predict_with_uncertainty(short_text)

    print("\nTRUE LABEL:", label_map[labels[i]])

    print("PREDICTED LABEL:", label_map[result["prediction"]])

    print("\nCONFIDENCE SCORE:", round(result["confidence"], 4))

    print("ENTROPY:", round(result["entropy"], 4))

    print("EMBEDDING VARIANCE:", round(result["embedding_variance"], 6))


# ============================================
# STEP 10: MULTI-RUN STABILITY TEST
# ============================================

print("\n")
print("="*60)
print("MULTI-RUN STABILITY TEST")
print("="*60)

test_text = "The movie was okay, not too bad but not amazing either."

for run in range(5):

    result = predict_with_uncertainty(test_text)

    print(f"\nRun {run+1}")

    print("Prediction:", label_map[result["prediction"]])

    print("Confidence:", round(result["confidence"], 4))

    print("Entropy:", round(result["entropy"], 4))


# ============================================
# STEP 11: INTERPRETATION
# ============================================

print("\n")
print("="*60)
print("INTERPRETATION GUIDE")
print("="*60)

print("""
1. High Confidence + Low Entropy
   -> Model is certain.

2. Low Confidence + High Entropy
   -> Model is uncertain.

3. High Embedding Variance
   -> Internal representations unstable.

4. Different outputs across runs
   -> High uncertainty.

This follows your teacher's instruction:
- Non-prompt uncertainty
- Multi-granular signals
- Hidden-state analysis
- Confidence estimation
""")